# Notebook – Faysal
## Project: Voorspelling Jeugdpopulatie Almere (CBS Wijken & Buurten)

**Doel van dit notebook (fase 1):**  
Controleren of de opgeschoonde dataset (df_v6_clean) inhoudelijk en technisch geschikt is voor analyse en modellering.

Binnen fase 1 controleer ik:
- Structuur en datatypes (Describe Data)
- Missende waarden en duplicaten (Verify Data Quality)
- Onlogische waarden (bijv. negatieve aantallen, jeugd > totaal inwoners)
- Indeling per jaar (extra aandacht: wijkindeling wijzigt vanaf 2022)
- Afleiden van eenvoudige ratio-variabelen (% jeugd t.o.v. totaal)
- Opslaan van een analyse-klare dataset voor de volgende fases

In [7]:
import pandas as pd
from pathlib import Path

path = Path("../../data/processed/df_v6_clean_final.csv")

df = pd.read_csv(path)
df.head()

,Peildatum,Wijk,AantalInwoners_5,k_0Tot15Jaar_8,k_15Tot25Jaar_9,Jaar
0,2018-01-01,WK003401,22790,3975,2420,2018
1,2019-01-01,WK003401,22875,3945,2415,2019
2,2020-01-01,WK003401,23275,4005,2485,2020
3,2021-01-01,WK003401,23530,4180,2465,2021
4,2022-01-01,WK003401,1635,175,150,2022


Ik laad de opgeschoonde dataset in vanuit data/processed. Met de check voorkom ik onduidelijke errors als het bestand of pad niet klopt.

In [8]:
df.shape, df.columns

((237, 6),
 Index(['Peildatum', 'Wijk', 'AantalInwoners_5', 'k_0Tot15Jaar_8',
        'k_15Tot25Jaar_9', 'Jaar'],
       dtype='str'))

Dit geeft het aantal rijen/kolommen en laat zien welke variabelen beschikbaar zijn voor het project.

In [9]:
df.info()
df.describe(include="all")
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 237 entries, 0 to 236
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Peildatum         237 non-null    str  
 1   Wijk              237 non-null    str  
 2   AantalInwoners_5  237 non-null    int64
 3   k_0Tot15Jaar_8    237 non-null    int64
 4   k_15Tot25Jaar_9   237 non-null    int64
 5   Jaar              237 non-null    int64
dtypes: int64(4), str(2)
memory usage: 11.2 KB


Peildatum           0
Wijk                0
AantalInwoners_5    0
k_0Tot15Jaar_8      0
k_15Tot25Jaar_9     0
Jaar                0
dtype: int64

df.info() controleert datatypes en of kolommen gevuld zijn.

df.describe() geeft basisstatistiek (min/max/mean) om uitschieters te herkennen.

isnull().sum() laat zien of er missende waarden zijn (belangrijk voor modelleren).
Dit past bij “Describe & Verify Data Quality” uit de Data Understanding-fase.

In [10]:
df.duplicated().sum()

np.int64(0)

Dubbele rijen kunnen tellingen vertekenen, dus ik controleer of records dubbel voorkomen. 

In [11]:
df["Peildatum"] = pd.to_datetime(df["Peildatum"], errors="coerce")
df["Jaar"] = df["Peildatum"].dt.year

df[["Peildatum", "Jaar"]].head()

,Peildatum,Jaar
0,2018-01-01,2018
1,2019-01-01,2019
2,2020-01-01,2020
3,2021-01-01,2021
4,2022-01-01,2022


Ik zet Peildatum om naar datetime en leid Jaar af. Dit maakt groeperen per jaar mogelijk en voorkomt errors bij groupby("Jaar"). 

In [12]:
kol_totaal = "AantalInwoners_5"
kol_0_15 = "k_0Tot15Jaar_8"
kol_15_25 = "k_15Tot25Jaar_9"

checks = {
    "negatief_0_15": (df[kol_0_15] < 0).sum(),
    "negatief_15_25": (df[kol_15_25] < 0).sum(),
    "negatief_totaal": (df[kol_totaal] < 0).sum(),
    "0_15_groter_dan_totaal": (df[kol_0_15] > df[kol_totaal]).sum(),
    "15_25_groter_dan_totaal": (df[kol_15_25] > df[kol_totaal]).sum(),
}

checks

{'negatief_0_15': np.int64(0),
 'negatief_15_25': np.int64(0),
 'negatief_totaal': np.int64(0),
 '0_15_groter_dan_totaal': np.int64(0),
 '15_25_groter_dan_totaal': np.int64(0)}

Aantallen inwoners en aantallen jongeren horen niet negatief te zijn. Ook kan een leeftijdsgroep niet groter zijn dan het totaal aantal inwoners in dezelfde wijk/peildatum. Dit is een eenvoudige inhoudelijke kwaliteitscontrole. 

In [13]:
df.groupby("Jaar")["Wijk"].nunique().sort_index()

Jaar
2018     5
2019     5
2020     5
2021     5
2022    53
2023    54
2024    55
2025    55
Name: Wijk, dtype: int64

Ik controleer hoeveel unieke wijken per jaar aanwezig zijn. Als dit rond 2022 verandert, kan dat invloed hebben op trendvergelijking en op hoe we het model opzetten. 

In [14]:
df["pct_0_15"] = (df[kol_0_15] / df[kol_totaal]) * 100
df["pct_15_25"] = (df[kol_15_25] / df[kol_totaal]) * 100

df[["Wijk", "Jaar", kol_totaal, kol_0_15, kol_15_25, "pct_0_15", "pct_15_25"]].head()

,Wijk,Jaar,AantalInwoners_5,k_0Tot15Jaar_8,k_15Tot25Jaar_9,pct_0_15,pct_15_25
0,WK003401,2018,22790,3975,2420,17.441860,10.618692
1,WK003401,2019,22875,3945,2415,17.245902,10.557377
2,WK003401,2020,23275,4005,2485,17.207304,10.676692
3,WK003401,2021,23530,4180,2465,17.764556,10.475988
4,WK003401,2022,1635,175,150,10.703364,9.174312


Naast absolute aantallen is het percentage jeugd per wijk nuttig, omdat wijken in totaal aantal inwoners verschillen. Dit is een eenvoudige afgeleide variabele (feature) voor latere analyse/modellering. 

In [15]:
df[["pct_0_15", "pct_15_25"]].describe()

,pct_0_15,pct_15_25
count,237.000000,237.000000
mean,18.277507,12.805460
std,6.034239,6.198061
min,2.083333,6.250000
25%,15.671642,10.135135
50%,17.287234,11.790393
75%,19.288646,14.258735
max,57.142857,83.333333


Hier kijk ik of percentages logisch zijn (bijv. geen negatieve waarden en geen extreem hoge waarden boven 100).

In [16]:
output_path = Path("../../data/processed/df_v8_ready_for_analysis.csv")
df.to_csv(output_path, index=False)

output_path.as_posix()

'../../data/processed/df_v8_ready_for_analysis.csv'

Ik sla een “analyse-klare” versie op, zodat we in de volgende fases (analyse/modellering) allemaal dezelfde kolommen en datatypes gebruiken.